# Importações

In [6]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from dotenv import load_dotenv

from sqlalchemy.engine import URL
from sqlalchemy import create_engine

load_dotenv()

SEED = 42

In [7]:
url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASS"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(url)

query = "SELECT * FROM gold.delivery_delay;"

In [8]:
df = pd.read_sql("SELECT * FROM gold.delivery_delay", engine)

df.shape

(96456, 22)

# Definindo treino e teste

In [ ]:
X = df.drop(columns=["order_id", "is_delayed"])
y = df["is_delayed"]

In [ ]:
# iremos dividir por data de corte
# onde o modelo é treinado com dados passados e testado com dados futuros.
# tendo em vista que is_delayed tem classes desbalanceadas
cutoff_date = pd.Timestamp("2018-06-01")

df["purchase_date"] = pd.to_datetime(df["purchase_date"])

train_df = df[df["purchase_date"] < cutoff_date].copy()
test_df = df[df["purchase_date"] >= cutoff_date].copy()

print("Treino:", train_df.shape, "\nTeste:", test_df.shape)

Treino: (77853, 22) 
Teste: (18603, 22)


In [17]:
print("Período treino:")
print(train_df["purchase_date"].min(), "→", train_df["purchase_date"].max())

print("\nPeríodo teste:")
print(test_df["purchase_date"].min(), "→", test_df["purchase_date"].max())

Período treino:
2016-09-15 00:00:00 → 2018-05-31 00:00:00

Período teste:
2018-06-01 00:00:00 → 2018-08-29 00:00:00


In [18]:
drop_cols = ["order_id", "is_delayed", "purchase_date"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["is_delayed"]

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["is_delayed"]

# Pré Processamento

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.preprocessing import OneHotEncoder

from sklearn.compose import ColumnTransformer

In [19]:
numeric_features = ["approval_time_hours", "estimated_delivery_days", "item_count", "unique_product_count",
                    "seller_count", "total_price", "total_freight", "total_weight_g", "total_volume_cm3",
                    "distance_km", "payment_count", "payment_type_count", "max_installments", "category_count"]

categorical_features = ["purchase_month", "purchase_day_of_week", "purchase_hour",
                        "shipping_state_type", "main_product_category",]

In [24]:
# criando os pipelines para pré-processamento dos dados numéricos e categóricos
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))])

In [25]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)])